# Water Quality

## Context

Access to safe drinking-water is essential to health, a basic human right and a component of effective policy for health protection. This is important as a health and development issue at a national, regional and local level. In some regions, it has been shown that investments in water supply and sanitation can yield a net economic benefit, since the reductions in adverse health effects and health care costs outweigh the costs of undertaking the interventions (acc:69%).

## Content

The water_potability.csv file contains water quality metrics for 3276 different water bodies.

1. Indicates if water is safe for human consumption where 1 means Potable and 0 means Not potable.

2. https://www.kaggle.com/datasets/adityakadiwal/water-potability

### Import necessary packages

In [1]:
import os
import numpy as np
import pandas as pd
# 显式导入 PPI 核心分析函数，拒绝 plot_cpp 签名缓存污染
from FL_cpp_method import analyze_dataset, plot_cpp
import utils  # 引入底层美学绘图路由模块

In [2]:
# ==============================================================================
# 1. 强力保底机制的多分类/二分类 Dirichlet 样本划分算法
# ==============================================================================
def dirichlet_multiclass_allocation(Y, Yhat, alpha_dir, num_clients, min_samples_per_client=50):
    """
    针对二分类/多分类离散标签的 Dirichlet 异质划分方案（带节点最低样本底线硬保障）。
    由于 water_potability 总样本量为 3276，20个节点下平均每节点 ~163个样本，
    此处将 min_samples_per_client 默认设为 50，以保证在极度 Non-IID 时不掏空节点。
    """
    classes = np.unique(Y)
    num_classes = len(classes)
    
    class_indices = {c: np.where(Y == c)[0] for c in classes}
    for c in classes:
        np.random.shuffle(class_indices[c])
        
    client_indices = [[] for _ in range(num_clients)]
    
    # 阶段一：各客户端分类均分保底，彻底破除空样本崩溃
    base_per_class = min_samples_per_client // num_classes
    if base_per_class < 1: base_per_class = 1
        
    for c in classes:
        indices = class_indices[c]
        available = len(indices)
        required = base_per_class * num_clients
        actual_base = base_per_class if required <= available else available // num_clients
        
        if actual_base > 0:
            for i in range(num_clients):
                client_indices[i].extend(indices[i * actual_base : (i + 1) * actual_base])
            class_indices[c] = indices[num_clients * actual_base:]
            
    # 阶段二：迪利克雷标签不平衡分配
    for c in classes:
        indices = class_indices[c]
        if len(indices) == 0: continue
            
        proportions = np.random.dirichlet([alpha_dir] * num_clients)
        counts = np.floor(proportions * len(indices)).astype(int)
        
        remainder = len(indices) - np.sum(counts)
        for _ in range(remainder):
            counts[np.random.choice(num_clients)] += 1
            
        start = 0
        for i in range(num_clients):
            end = start + counts[i]
            client_indices[i].extend(indices[start:end])
            start = end
            
    # 打乱并组装各客户端最终序列
    reordered_indices = []
    actual_sizes = []
    for i in range(num_clients):
        np.random.shuffle(client_indices[i])
        reordered_indices.extend(client_indices[i])
        actual_sizes.append(len(client_indices[i]))
        
    return Y[reordered_indices], Yhat[reordered_indices], actual_sizes

In [3]:
# %%
# ==============================================================================
# 1. 实验控制元参数初始化（水质可饮用性分析：alpha=0.05 对应 95% 置信区间）
# ==============================================================================
dataset_name = 'water_potability'
alpha = 0.05  
method = "mean"
num_clients = 20  

xlim = [0, 1.1]  
ylim = [0, 1.0]

iid_total_records = []
non_iid_total_records = []
acc_steps = np.arange(0.1, 1.1, 0.1)

# ==============================================================================
# 2. 纵向多精度大循环核心
# ==============================================================================
for acc in acc_steps:
    acc_str = f"{acc:.1f}"
    data_path = f'../data/{dataset_name}/{dataset_name}_acc_{acc_str}.npz'
    
    if not os.path.exists(data_path):
        print(f"⚠️ [跳过] 未检测到精度阶梯文件: {data_path}")
        continue
        
    print(f"\n⚡ [当前进度] 正在全面计算精度级别 -> Acc = {acc_str}")
    data = np.load(data_path)
    Y_total = data["Y"]
    Yhat_total = data["Y_hat"] if "Y_hat" in data.files else data["Yhat"]
    
    # --------------------------------------------------------------------------
    # 3.1 运行当前精度下的标准 IID 实验并画图
    # --------------------------------------------------------------------------
    dataset_dist_iid = 'IID'
    num_ratio_iid = [1] * num_clients
    
    true_theta_iid, cpp_intervals_iid, ppi_ci_combined_iid, mean_cpp_iid = analyze_dataset(
        alpha, None, Y_total, Yhat_total, dataset_dist_iid, num_ratio_iid, method, grid=None
    )
    title_iid = f"Acc = {acc_str}"
    plot_cpp(true_theta_iid, cpp_intervals_iid, ppi_ci_combined_iid, mean_cpp_iid, 
             dataset_name, dataset_dist_iid, acc_str, None, xlim, ylim, title_iid, None)
    
    iid_total_records.append({
        'dataset': dataset_name, 'target_accuracy': acc_str, 'distribution': dataset_dist_iid, 'alpha_dir': 'None',
        'ppi_ci_lower': ppi_ci_combined_iid[0], 'ppi_ci_upper': ppi_ci_combined_iid[1], 'mean_cpp_lower': mean_cpp_iid[0], 'mean_cpp_upper': mean_cpp_iid[1]
    })
    
    # --------------------------------------------------------------------------
    # 3.2 运行当前精度下的三种 alpha_Dir Non-IID 实验并画图
    # --------------------------------------------------------------------------
    alpha_dir_list = [1.0, 0.1, 0.01]
    dataset_dist_non = 'Non-IID'
    for alpha_dir in alpha_dir_list:
        Y_dir, Yhat_dir, actual_sizes = dirichlet_multiclass_allocation(
            Y_total, Yhat_total, alpha_dir, num_clients, min_samples_per_client=50
        )
        true_theta_dir, cpp_intervals_dir, ppi_ci_combined_dir, mean_cpp_dir = analyze_dataset(
            alpha, None, Y_dir, Yhat_dir, 'pre_split', actual_sizes, method, grid=None
        )
        # 🛠️ 核心更正：将当前变化的 Acc 精度显式焊接到 Non-IID 标签的括号中
        title_dir = f"$\\alpha_{{Dir}} = {alpha_dir}$ (Acc = {acc_str})"
        plot_cpp(true_theta_dir, cpp_intervals_dir, ppi_ci_combined_dir, mean_cpp_dir, 
                 dataset_name, dataset_dist_non, acc_str, alpha_dir, xlim, ylim, title_dir, None)
        
        non_iid_total_records.append({
            'dataset': dataset_name, 'target_accuracy': acc_str, 'distribution': dataset_dist_non, 'alpha_dir': str(alpha_dir),
            'ppi_ci_lower': ppi_ci_combined_dir[0], 'ppi_ci_upper': ppi_ci_combined_dir[1], 'mean_cpp_lower': mean_cpp_dir[0], 'mean_cpp_upper': mean_cpp_dir[1]
        })

# 跨精度维度多合一 Master CSV 归档
csv_flat_dir = os.path.join('.', 'result', dataset_name, 'csv')
os.makedirs(csv_flat_dir, exist_ok=True)
if len(iid_total_records) > 0:
    pd.DataFrame(iid_total_records).to_csv(os.path.join(csv_flat_dir, f'{dataset_name}_IID_summary.csv'), index=False)
if len(non_iid_total_records) > 0:
    pd.DataFrame(non_iid_total_records).to_csv(os.path.join(csv_flat_dir, f'{dataset_name}_Non-IID_summary.csv'), index=False)


⚡ [当前进度] 正在全面计算精度级别 -> Acc = 0.1
labeled_ratio 0.3
分组： 1
带标签的样本量： 49
不带标签的样本量： 115
分组： 2
带标签的样本量： 49
不带标签的样本量： 115
分组： 3
带标签的样本量： 49
不带标签的样本量： 115
分组： 4
带标签的样本量： 49
不带标签的样本量： 115
分组： 5
带标签的样本量： 49
不带标签的样本量： 115
分组： 6
带标签的样本量： 49
不带标签的样本量： 115
分组： 7
带标签的样本量： 49
不带标签的样本量： 115
分组： 8
带标签的样本量： 49
不带标签的样本量： 115
分组： 9
带标签的样本量： 49
不带标签的样本量： 115
分组： 10
带标签的样本量： 49
不带标签的样本量： 115
分组： 11
带标签的样本量： 49
不带标签的样本量： 115
分组： 12
带标签的样本量： 49
不带标签的样本量： 115
分组： 13
带标签的样本量： 49
不带标签的样本量： 115
分组： 14
带标签的样本量： 49
不带标签的样本量： 115
分组： 15
带标签的样本量： 49
不带标签的样本量： 115
分组： 16
带标签的样本量： 49
不带标签的样本量： 115
分组： 17
带标签的样本量： 48
不带标签的样本量： 115
分组： 18
带标签的样本量： 48
不带标签的样本量： 115
分组： 19
带标签的样本量： 48
不带标签的样本量： 115
分组： 20
带标签的样本量： 48
不带标签的样本量： 115
带标签的样本量： 976
不带标签的样本量： 2300

最终结果：
真实 theta: 0.3901098901098901
CPP intervals: [array([0.22235015, 0.40268908]), array([0.26168703, 0.48365466]), array([0.24369759, 0.46531804]), array([0.2781696 , 0.49107341]), array([0.25639882, 0.4438843 ]), array([0.27295255, 0.45798742]), array([0.27720372, 

In [4]:
# %%
# ==============================================================================
# 1. 固定准确率为 50% 的基准数据集调入
# ==============================================================================
fixed_acc_str = '0.5'
data_path = f'../data/{dataset_name}/{dataset_name}_acc_{fixed_acc_str}.npz'

print(f"正在读取固定 50% 准确率的目标基准数据: {data_path}")
data = np.load(data_path)
Y_total = data["Y"]
Yhat_total = data["Y_hat"] if "Y_hat" in data.files else data["Yhat"]

iid_ratio_records = []
non_iid_ratio_records = []
ratio_steps = [0.1, 0.2, 0.3, 0.4, 0.5]

# ==============================================================================
# 2. 纵向标签比例变动大循环核心
# ==============================================================================
for ratio in ratio_steps:
    ratio_str = f"{ratio:.1f}"
    sub_folder_name = f"ratio_{ratio_str}"  
    print(f"\n🚀 [实验进行中] 正在注入比例阶梯 -> labeled_ratio = {ratio_str}")
    
    # 4.1 变比率 IID 支线
    dataset_dist_iid = 'IID'
    num_ratio_iid = [1] * num_clients
    true_theta_iid, cpp_intervals_iid, ppi_ci_combined_iid, mean_cpp_iid = analyze_dataset(
        alpha, None, Y_total, Yhat_total, dataset_dist_iid, num_ratio_iid, method, None, current_ratio=ratio
    )
    # 🛠️ 核心更正：IID 支线也清晰注明当前的比例变量 \lambda
    title_iid = f"$\\lambda = {ratio_str}$ (Acc = 0.5)"
    plot_cpp(true_theta_iid, cpp_intervals_iid, ppi_ci_combined_iid, mean_cpp_iid, 
             dataset_name, dataset_dist_iid, fixed_acc_str, None, xlim, ylim, title_iid, sub_folder_name)
    
    iid_ratio_records.append({
        'dataset': dataset_name, 'fixed_accuracy': fixed_acc_str, 'labeled_ratio': ratio_str, 'distribution': dataset_dist_iid, 'alpha_dir': 'None',
        'ppi_ci_lower': ppi_ci_combined_iid[0], 'ppi_ci_upper': ppi_ci_combined_iid[1], 'mean_cpp_lower': mean_cpp_iid[0], 'mean_cpp_upper': mean_cpp_iid[1]
    })
    
    # 4.2 变比率 Non-IID 支线
    dataset_dist_non = 'Non-IID'
    alpha_dir_list = [1.0, 0.1, 0.01]
    for alpha_dir in alpha_dir_list:
        Y_dir, Yhat_dir, actual_sizes = dirichlet_multiclass_allocation(
            Y_total, Yhat_total, alpha_dir, num_clients, min_samples_per_client=50
        )
        true_theta_dir, cpp_intervals_dir, ppi_ci_combined_dir, mean_cpp_dir = analyze_dataset(
            alpha, None, Y_dir, Yhat_dir, 'pre_split', actual_sizes, method, grid=None, current_ratio=ratio
        )
        # 🛠️ 核心更正：将变化的少样本抽样比 \lambda 优雅追加到 alpha_Dir 的后侧
        title_dir = f"$\\alpha_{{Dir}} = {alpha_dir}$ ($\\lambda = {ratio_str}$)"
        plot_cpp(true_theta_dir, cpp_intervals_dir, ppi_ci_combined_dir, mean_cpp_dir, 
                 dataset_name, dataset_dist_non, fixed_acc_str, alpha_dir, xlim, ylim, title_dir, sub_folder_name)
        
        non_iid_ratio_records.append({
            'dataset': dataset_name, 'fixed_accuracy': fixed_acc_str, 'labeled_ratio': ratio_str, 'distribution': dataset_dist_non, 'alpha_dir': str(alpha_dir),
            'ppi_ci_lower': ppi_ci_combined_dir[0], 'ppi_ci_upper': ppi_ci_combined_dir[1], 'mean_cpp_lower': mean_cpp_dir[0], 'mean_cpp_upper': mean_cpp_dir[1]
        })

# 跨比率维度多合一 Ratio CSV 写出
if len(iid_ratio_records) > 0:
    pd.DataFrame(iid_ratio_records).to_csv(os.path.join(csv_flat_dir, f'{dataset_name}_IID_ratio_summary.csv'), index=False)
if len(non_iid_ratio_records) > 0:
    pd.DataFrame(non_iid_ratio_records).to_csv(os.path.join(csv_flat_dir, f'{dataset_name}_Non-IID_ratio_summary.csv'), index=False)

正在读取固定 50% 准确率的目标基准数据: ../data/water_potability/water_potability_acc_0.5.npz

🚀 [实验进行中] 正在注入比例阶梯 -> labeled_ratio = 0.1
labeled_ratio 0.1
分组： 1
带标签的样本量： 16
不带标签的样本量： 148
分组： 2
带标签的样本量： 16
不带标签的样本量： 148
分组： 3
带标签的样本量： 16
不带标签的样本量： 148
分组： 4
带标签的样本量： 16
不带标签的样本量： 148
分组： 5
带标签的样本量： 16
不带标签的样本量： 148
分组： 6
带标签的样本量： 16
不带标签的样本量： 148
分组： 7
带标签的样本量： 16
不带标签的样本量： 148
分组： 8
带标签的样本量： 16
不带标签的样本量： 148
分组： 9
带标签的样本量： 16
不带标签的样本量： 148
分组： 10
带标签的样本量： 16
不带标签的样本量： 148
分组： 11
带标签的样本量： 16
不带标签的样本量： 148
分组： 12
带标签的样本量： 16
不带标签的样本量： 148
分组： 13
带标签的样本量： 16
不带标签的样本量： 148
分组： 14
带标签的样本量： 16
不带标签的样本量： 148
分组： 15
带标签的样本量： 16
不带标签的样本量： 148
分组： 16
带标签的样本量： 16
不带标签的样本量： 148
分组： 17
带标签的样本量： 16
不带标签的样本量： 147
分组： 18
带标签的样本量： 16
不带标签的样本量： 147
分组： 19
带标签的样本量： 16
不带标签的样本量： 147
分组： 20
带标签的样本量： 16
不带标签的样本量： 147
带标签的样本量： 320
不带标签的样本量： 2956

最终结果：
真实 theta: 0.3901098901098901
CPP intervals: [array([0.22875744, 0.6123263 ]), array([0.12950815, 0.51135372]), array([0.21422073, 0.6297174 ]), array([0.29651484, 0.74197897]),